# Maps for analysis and manuscript (cleaned)

Use this to generate various maps. Will require joining various datasets.

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

from collections import defaultdict

import datetime
import os

import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
metadata_dir = "metadata"

## Harmonize country names

In [ ]:
# Import UN jurisdisction data from https://unstats.un.org/unsd/methodology/m49/overview/
# Data sources: United Nations (2025) – with major processing by Our World in Data
# https://ourworldindata.org/grapher/united-nations-membership-status?overlay=download-data
# Use this spreadsheet as the basis for filtering
# It's mainly for convenience since I can join it with existing geopandas data
unsd_df = pd.read_csv(os.path.join(metadata_dir, "UNSD_Methodology.csv"), delimiter=";")

un_membership_df = pd.read_csv(os.path.join(metadata_dir, "united-nations-membership-status.csv"))
un_membership_df = un_membership_df[un_membership_df["Year"]==2024]

# Add Palestine as observer. Later on we add Cook Islands and Taiwan as "Associate" and "Former"
un_data = un_membership_df[(un_membership_df["United Nations membership status"]=="Member") | (un_membership_df["Entity"]=="Palestine")]
un_data.loc[un_data["Entity"]=="Palestine","United Nations membership status"] = "Observer"

In [ ]:
# Mapping file
url = os.path.join(metadata_dir, "ne_50m_admin_0_countries.zip")

world = gpd.read_file(url)
world_reduced = world[["SOVEREIGNT","SOV_A3","ADMIN","geometry"]] # Reduced to just a few variables (sovereign, codes, sub-region names, geometry)

In [ ]:
# Create a correspondance between UN and NE country names
world_reduced["Country_UN"] = world_reduced["ADMIN"]

world_reduced.loc[world_reduced["Country_UN"]=="The Bahamas","Country_UN"] = "Bahamas"
world_reduced.loc[world_reduced["Country_UN"]=="Republic of the Congo","Country_UN"] = "Congo"
world_reduced.loc[world_reduced["Country_UN"]=="United Republic of Tanzania","Country_UN"] = "Tanzania"
world_reduced.loc[world_reduced["Country_UN"]=="Democratic Republic of the Congo","Country_UN"] = "Democratic Republic of Congo"
world_reduced.loc[world_reduced["Country_UN"]=="United States of America","Country_UN"] = "United States"
world_reduced.loc[world_reduced["Country_UN"]=="eSwatini","Country_UN"] = "Eswatini"
world_reduced.loc[world_reduced["Country_UN"]=="Ivory Coast","Country_UN"] = "Cote d'Ivoire"
world_reduced.loc[world_reduced["Country_UN"]=="Federated States of Micronesia","Country_UN"] = "Micronesia (country)"
world_reduced.loc[world_reduced["Country_UN"]=="Republic of Serbia","Country_UN"] = "Serbia"
world_reduced.loc[world_reduced["Country_UN"]=="São Tomé and Principe","Country_UN"] = "Sao Tome and Principe"
world_reduced.loc[world_reduced["Country_UN"]=="Cabo Verde","Country_UN"] = "Cape Verde"

In [ ]:
# Wikipedia demonyms
demonym_data = pd.read_csv("../metadata/demonyms_with_keywords_202605.csv")

In [ ]:
demonym_data["Country_UN"] = demonym_data["country"]

In [ ]:
demonym_data.loc[demonym_data["Country_UN"]=="United Kingdom of Great Britain and Northern Ireland","Country_UN"] = "United Kingdom"
demonym_data.loc[demonym_data["Country_UN"]=="Federated States of Micronesia","Country_UN"] = "Micronesia (country)"
demonym_data.loc[demonym_data["Country_UN"]=="United States of America","Country_UN"] = "United States"
demonym_data.loc[demonym_data["Country_UN"]=="Czech Republic","Country_UN"] = "Czechia"
demonym_data.loc[demonym_data["Country_UN"]=="Federated States of Micronesia","Country_UN"] = "Micronesia (country)"
demonym_data.loc[demonym_data["Country_UN"]=="The Bahamas","Country_UN"] = "Bahamas"
demonym_data.loc[demonym_data["Country_UN"]=="Republic of Ireland","Country_UN"] = "Ireland"
demonym_data.loc[demonym_data["Country_UN"]=="State of Palestine","Country_UN"] = "Palestine"

In [ ]:
demonym_data.rename(columns={
    "revision_id":"revision_id_denomym",
    "page_name":"page_name_demonym",
    "Country":"country_demonym",
    "kw_flag":"kw_flag_demonym",
    "kw_section_flag":"kw_section_flag_demonym"
                            }, inplace=True)

In [ ]:
demographics_data = pd.read_csv("../metadata/demographics_with_keywords_202602.csv")

In [ ]:
demographics_data["Country_UN"] = demographics_data["country"]

In [ ]:
demographics_data.loc[demographics_data["Country_UN"]=="Georgia (country)","Country_UN"] = "Georgia"
demographics_data.loc[demographics_data["Country_UN"]=="Republic of Ireland","Country_UN"] = "Ireland"
demographics_data.loc[demographics_data["Country_UN"]=="Federated States of Micronesia","Country_UN"] = "Micronesia (country)"
demographics_data.loc[demographics_data["Country_UN"]=="Democratic Republic of the Congo","Country_UN"] = "Democratic Republic of Congo"
demographics_data.loc[demographics_data["Country_UN"]=="Republic of the Congo","Country_UN"] = "Congo"
demographics_data.loc[demographics_data["Country_UN"]=="Czech Republic","Country_UN"] = "Czechia"
demographics_data.loc[demographics_data["Country_UN"]=="Ivory Coast","Country_UN"] = "Cote d'Ivoire"
demographics_data.loc[demographics_data["Country_UN"]=="São Tomé and Príncipe","Country_UN"] = "Sao Tome and Principe"
demographics_data.loc[demographics_data["Country_UN"]=="Timor-Leste","Country_UN"] = "East Timor"

In [ ]:
demographics_data.rename(columns={
    "revision_id":"revision_id_demographics",
    "kw_flag":"kw_flag_demographics",
    "kw_section_flag":"kw_section_flag_demographics",
    "page_name":"page_name_demographics",
    "country":"country_demographics"
    }, inplace=True
                        )

## Mapping mentions of genetics


In [ ]:
# Create a dataframe with the three we've tidied up
world_mapping = pd.merge(
    left = world_reduced,
    right = un_data[["Entity","Code","United Nations membership status"]],
    how = "left",
    left_on = "Country_UN",
    right_on = "Entity"
)

# Merge demonym data
world_mapping = world_mapping.drop("Entity", axis=1)
world_mapping = pd.merge(
    left = world_mapping,
    right = demonym_data,
    how = "left",
    left_on = "Country_UN",
    right_on = "Country_UN"
)

# Merge demographics data
world_mapping = pd.merge(
    left = world_mapping,
    right = demographics_data,
    how = "left",
    left_on = "Country_UN",
    right_on = "Country_UN"
)

In [ ]:
# Change column 6 to "Former" or "Associate"
world_mapping.loc[world_mapping["ADMIN"] == "Cook Islands", "United Nations membership status"] = "Associate"
world_mapping.loc[world_mapping["ADMIN"] == "Taiwan", "United Nations membership status"] = "Former"

In [ ]:
# Reduce to those in UN/observer/former/associate states
world_mapping_un = world_mapping[np.isin(world_mapping["United Nations membership status"].values,["Member","Observer","Associate","Former"])]

In [ ]:
world_mapping_un.loc[:,"demonym_genetics"] = world_mapping_un["kw_flag_demonym"]
world_mapping_un.loc[:,"demographics_genetics"] = world_mapping_un["kw_flag_demographics"]

world_mapping_un.loc[:,"demonym_genetics_section"] = world_mapping_un["kw_section_flag_demonym"]
world_mapping_un.loc[:,"demographics_genetics_section"] = world_mapping_un["kw_section_flag_demographics"]

In [ ]:
# Keyword presence
world_mapping_un.loc[world_mapping_un["demonym_genetics"].isna(),"demonym_genetics"] = "No demonym page"
world_mapping_un.loc[world_mapping_un["demonym_genetics"]==True,"demonym_genetics"] = "Genetics mentioned"
world_mapping_un.loc[world_mapping_un["demonym_genetics"]==False,"demonym_genetics"] = "Genetics not mentioned"

# Demographics pages
world_mapping_un.loc[world_mapping_un["demographics_genetics"]==True,"demographics_genetics"] = "Genetics mentioned"
world_mapping_un.loc[world_mapping_un["demographics_genetics"]==False,"demographics_genetics"] = "Genetics not mentioned"

# Demographics OR demonyms mention demographics
world_mapping_un.loc[:,"demonym_demographics_genetics"] = ""
world_mapping_un.loc[
    (world_mapping_un["demonym_genetics"]=="Genetics mentioned") | (world_mapping_un["demographics_genetics"]=="Genetics mentioned"),
    "demonym_demographics_genetics"
    ] = "Genetics mentioned"
world_mapping_un.loc[
    ~((world_mapping_un["demonym_genetics"]=="Genetics mentioned") | (world_mapping_un["demographics_genetics"]=="Genetics mentioned")),
    "demonym_demographics_genetics"
    ] = "Genetics not mentioned"

# Set up sorting for the figure legends and colours
world_mapping_un["demonym_genetics"] = pd.Categorical(world_mapping_un["demonym_genetics"],
                                                                  ["Genetics mentioned","Genetics not mentioned","No demonym page"])
world_mapping_un["demographics_genetics"] = pd.Categorical(world_mapping_un["demographics_genetics"],
                                                                  ["Genetics mentioned","Genetics not mentioned"])
world_mapping_un["demonym_demographics_genetics"] = pd.Categorical(world_mapping_un["demonym_demographics_genetics"],
                                                                  ["Genetics mentioned","Genetics not mentioned"])

In [ ]:
# Genetics section presence

world_mapping_un.loc[world_mapping_un["demonym_genetics_section"].isna(),"demonym_genetics_section"] = "No demonym page"
world_mapping_un.loc[world_mapping_un["demonym_genetics_section"]==True,"demonym_genetics_section"] = "Has genetics section"
world_mapping_un.loc[world_mapping_un["demonym_genetics_section"]==False,"demonym_genetics_section"] = "No genetics section"

# Demographics pages
world_mapping_un.loc[world_mapping_un["demographics_genetics_section"]==True,"demographics_genetics_section"] = "Has genetics section"
world_mapping_un.loc[world_mapping_un["demographics_genetics_section"]==False,"demographics_genetics_section"] = "No genetics section"

# Demographics OR demonyms mention demographics
world_mapping_un.loc[:,"demonym_demographics_genetics_section"] = ""
world_mapping_un.loc[
    (world_mapping_un["demonym_genetics_section"]=="Has genetics section") | (world_mapping_un["demographics_genetics_section"]=="Has genetics section"),
    "demonym_demographics_genetics_section"
    ] = "Has genetics section"
world_mapping_un.loc[
    ~((world_mapping_un["demonym_genetics_section"]=="Has genetics section") | (world_mapping_un["demographics_genetics_section"]=="Has genetics section")),
    "demonym_demographics_genetics_section"
    ] = "No genetics section"

# Set up sorting for the figure legends and colours
world_mapping_un["demonym_genetics_section"] = pd.Categorical(world_mapping_un["demonym_genetics_section"],
                                                                  ["Has genetics section","No genetics section","No demonym page"])
world_mapping_un["demographics_genetics_section"] = pd.Categorical(world_mapping_un["demographics_genetics_section"],
                                                                  ["Has genetics section","No genetics section"])
world_mapping_un["demonym_demographics_genetics_section"] = pd.Categorical(world_mapping_un["demonym_demographics_genetics_section"],
                                                                  ["Has genetics section","No genetics section"])

In [ ]:
test_cmap = mpl.colors.ListedColormap(["#496831", "#EFE287", "#E6E0DB"]) # three colours for demonyms
cmap_demographics = mpl.colors.ListedColormap(["#496831", "#EFE287"]) # Only two colours since every country has a page

In [ ]:
fig = plt.figure(figsize = (10,6))

ax = world_mapping_un.plot(
    column="demonym_genetics",
    figsize = (18,9),
    legend = True,
    legend_kwds={"loc": "lower center"},
    cmap=test_cmap,
    edgecolor="#000000",
    linewidth=0.5
)

ax.set_xticks([],[])
ax.set_yticks([],[])

#plt.savefig("figures/demonyms_world_map.svg", bbox_inches="tight", dpi=300)

In [ ]:
# Animate the map above
from PIL import Image
from matplotlib.patches import Patch

steps = [
    ("No demonym page", "#E6E0DB"),
    ("Genetics not mentioned", "#EFE287"),
    ("Genetics mentioned", "#496831"),
]

minx, miny, maxx, maxy = world_mapping_un.total_bounds
w = 20
h = w * (maxy - miny) / (maxx - minx)

fig, ax = plt.subplots(figsize=(w, h), dpi=150)
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

def render(frame):
    ax.clear()
    ax.set_axis_off()
    world_mapping_un.plot(ax=ax, color="#FFFFFF", edgecolor="#000000",
                          linewidth=0.5, aspect=None)

    handles = []
    for i, (cat, colour) in enumerate(steps):
        if i < frame:
            subset = world_mapping_un[world_mapping_un["demonym_genetics"] == cat]
            if not subset.empty:
                subset.plot(ax=ax, color=colour, edgecolor="#000000",
                            linewidth=0.5, aspect=None)
            handles.append(Patch(facecolor=colour, edgecolor="#000000", label=cat))
        else:
            handles.append(Patch(facecolor="none", edgecolor="none", label=" " * len(cat)))

    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.02), framealpha=0.9)

    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_aspect("equal")

    fig.canvas.draw()
    return Image.fromarray(np.asarray(fig.canvas.buffer_rgba()).copy()).convert("RGB")

frames = [render(f) for f in range(len(steps) + 1)]
plt.close(fig)

# Output individual frames
# For vector versions instead, 
# re-call render(f) but swap the return for fig.savefig(f"figures/demonyms_world_map_frame_{f}.svg", bbox_inches="tight")
for i, img in enumerate(frames):
    img.save(f"figures/demonyms_world_map_frame_{i}.png")

frames[0].save(
    "figures/demonyms_world_map.gif",
    save_all=True,
    append_images=frames[1:],
    duration=[3000, 3000, 3000, 4000],
    disposal=2,
)

Sections below

In [ ]:
cmap_demographics = mpl.colors.ListedColormap(["#496831", "#EFE287"])

fig = plt.figure(figsize = (10,6))
#ax = fig.add_subplot(111)

ax = world_mapping_un.plot(
    column="demographics_genetics",
    figsize = (18,9),
    legend = True,
    legend_kwds={"loc": "lower center"},
    cmap=cmap_demographics,
    #cmap=test_cmap,
    edgecolor="#000000",
    linewidth=0.5
)

ax.set_xticks([],[])
ax.set_yticks([],[])

#plt.savefig("figures/demographics_world_map.svg", bbox_inches="tight", dpi=300)

In [ ]:
# Demographics and demonyms
cmap_genetics = mpl.colors.ListedColormap(["#496831", "#EFE287"])

fig = plt.figure(figsize = (10,6))

ax = world_mapping_un.plot(
    column="demonym_demographics_genetics",
    figsize = (18,9),
    legend = True,
    legend_kwds={"loc": "lower center"},
    cmap=cmap_genetics,
    edgecolor="#000000",
    linewidth=0.5
)

ax.set_xticks([],[])
ax.set_yticks([],[])

#plt.savefig("figures/demographics_and_demonyms_world_map.png", bbox_inches="tight", dpi=300)

In [ ]:
fig = plt.figure(figsize = (10,6))
#ax = fig.add_subplot(111)

ax = world_mapping_un.plot(
    column="demonym_genetics_section",
    figsize = (18,9),
    legend = True,
    legend_kwds={"loc": "lower center"},
    cmap=test_cmap,
    edgecolor="#000000",
    linewidth=0.5
)

ax.set_xticks([],[])
ax.set_yticks([],[])

#plt.savefig("figures/demonyms_section_world_map.png", bbox_inches="tight", dpi=300)

In [ ]:
fig = plt.figure(figsize = (10,6))

ax = world_mapping_un.plot(
    column="demographics_genetics_section",
    figsize = (18,9),
    legend = True,
    legend_kwds={"loc": "lower center"},
    cmap=cmap_demographics,
    #cmap = test_cmap,
    edgecolor="#000000",
    linewidth=0.5
)

ax.set_xticks([],[])
ax.set_yticks([],[])

#plt.savefig("figures/demographics_section_world_map.svg", bbox_inches="tight", dpi=300)

In [ ]:
fig = plt.figure(figsize = (10,6))

ax = world_mapping_un.plot(
    column="demonym_demographics_genetics_section",
    figsize = (18,9),
    legend = True,
    legend_kwds={"loc": "lower center"},
    cmap=cmap_demographics,
    #cmap=test_cmap,
    edgecolor="#000000",
    linewidth=0.5
)

ax.set_xticks([],[])
ax.set_yticks([],[])

#plt.savefig("figures/demonyms_demographics_section_world_map.svg", bbox_inches="tight", dpi=300)